In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json, os, sys
import pandas as pd
import datetime as dt

from rockyclickup.wrapper import Session as RCUSession
from rockyclickup.utils import response_to_dataframe as rcu_res_to_df, datetime_nearest_day
from rockyclickup.models import MODEL_LOOKUP, Client, FSA, DCA, HSA, HRA, PKG, TRN, LSA, ADO, EDU

from rockyelevate.wrapper import Session as ELVSession
from rockyelevate.utils import response_to_dataframe as elv_res_to_df

from rockydb.connection import CoreDB

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.merger import Merger 


In [3]:
rcu = RCUSession()
elv = ELVSession("PROD", multithread=True, max_threads=40)
rdb = CoreDB()

In [4]:
# all_entities = rdb.get_all_from_table(table="entity")
# org_ids = [e.elevate_id for e in all_entities if e.elevate_id]

In [5]:
# plans = elv.get_plans_by_org(oids=set(org_ids), detail=True)

In [6]:
# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_plans.json", "w") as f:
#     json.dump(plans, f , indent=4)

In [ ]:
# cu_plan_responses = []
# for model in [FSA, DCA, HSA, HRA, PKG, TRN, LSA, ADO, EDU]:
#     res = rcu.get_full_list(model=model)

#     cu_plan_responses.extend(res)

# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_cu_plans.json", "w") as f:
#     json.dump(cu_plan_responses, f, indent=4)

In [8]:
# client_res = rcu.get_full_list(model=Client)

# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_clients.json", "w") as f:
#     json.dump(client_res, f, indent=4)


In [9]:
# oids = elv_plan_df['organization_id'].to_list()
# # print(len(oids))
# oids_list = list(set(list(oids)))
# print(oids_list[0:10])
# org_res = elv.get_organizations(oids=oids_list, details=['STATUS', 'PARTNER_EXTERNAL_IDENTIFIER'], types=["COMPANY"])
# # with open(f"")

# with open(f"20251112_org_res.json", "w") as f:
#     json.dump(org_res, f, indent=4)

In [95]:
# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_plans.json", "r") as f:
#     elv_plans = json.load(f)

# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_cu_plans.json", "r") as f:
#     cu_plan_responses = json.load(f)

# with open(f"{dt.datetime.now().strftime("%Y%m%d")}_clients.json", "r") as f:
#     client_res = json.load(f)

with open(f"20251112_org_res.json", "r") as f:
    org_res = json.load(f)

with open(f"20251209_plans.json", "r") as f:
    elv_plans = json.load(f)

with open(f"20251211_cu_plans.json", "r") as f:
    cu_plan_responses = json.load(f)

with open(f"20251209_clients.json", "r") as f:
    client_res = json.load(f)

org_df = elv_res_to_df(org_res)
elv_plan_df = elv_res_to_df(elv_plans)
rcu_plan_df = rcu_res_to_df(cu_plan_responses)
client_df = rcu_res_to_df(client_res)

Field not in config.db: return_funds_after_closure e6815ce8-443f-435f-a312-47c650a21abc
Field not in config.db: structure 5515a85d-d6fb-45bf-8e40-081e41b5e11b
Field not in config.db: balance_availability_model e4cc3d89-8316-4e20-8c95-5e2f9b0c5a7e
Field not in config.db: termination_vendor_moved_to 09d7944a-ec00-4485-baf8-b481bbb16402
Field not in config.db: update_account_managers 702d85f6-7155-447c-9019-206db17ab27c
Field not in config.db: termination_date_client 3bdbb61f-341d-4699-8595-02a79f1b9e96
Field not in config.db: fein c5f85b6b-a5b7-4782-8d4d-f2979d7d7fea
Field not in config.db: naics_code b9e909aa-bf7c-4fdf-b252-693dd7d6df75
Field not in config.db: naics_national_industry ba05044f-1458-417f-8faf-f92bf99e8343
Field not in config.db: naics_naics_industry 5be04b2f-d4cc-4c03-9e26-422206b4e620
Field not in config.db: naics_title d7ec88f6-8f86-4bd2-be35-70186fb6490c
Field not in config.db: naics_subsector 6ba84f65-7eeb-4e01-981d-6767a8747fbf
Field not in config.db: naics_sector 47

In [96]:
elv_plans = elv_plan_df.copy()
rcu_plans = rcu_plan_df.copy()
clients = client_df.copy()
orgs = org_df.copy()


merger = Merger(
    client_df=clients,
    cu_plan_df = rcu_plan_df,
    elv_plan_df = elv_plans,
    organization_df = orgs
)

True
True


In [97]:
merge_df = merger.merge_all()

['parent_id', 'organization_path']


In [98]:
compare_df = merge_df.copy()

compare_df[[
    'plan_coverage_config.grace_period_type.grace_period_days_amount',
    'plan_coverage_config.grace_period_type.grace_period_type'
]]

compare_df["grace_period_fix"] = compare_df['plan_coverage_config.grace_period_type.grace_period_days_amount']

compare_df['grace_period_fix'] = compare_df['grace_period_fix'].fillna(
    compare_df['plan_coverage_config.grace_period_type.grace_period_type'].apply(
        lambda x:
        75 if x == "TWO_AND_HALF_MONTH"
        else 0
    )
).astype(int).astype(str)

compare_df['grace_period'] = compare_df['grace_period'].apply(lambda x: x if pd.notna(x) else 0).astype(str)

grace_periods_to_fix = compare_df[
    (compare_df['grace_period'] != compare_df['grace_period_fix']) &
    (compare_df['cu_plan_id'].notna()) &
    (compare_df['elv_plan_id'].notna())
]

dc_df = elv_plan_df.copy()
dc_df['double_check'] = dc_df.apply(
    lambda row:
        75 if row['plan_coverage_config.grace_period_type.grace_period_type'] == "TWO_AND_HALF_MONTH" else
        row['plan_coverage_config.grace_period_type.grace_period_days_amount'] if row['plan_coverage_config.grace_period_type.grace_period_type'] == "CUSTOM" else
        0 if row['plan_coverage_config.grace_period_type.grace_period_type'] == "DOES_NOT_APPLY"
        else pd.NA,
    axis=1
)

grace_period_map = {r['id']: r['double_check'] for _, r in dc_df.iterrows()}

grace_periods_to_fix['double_check'] = grace_periods_to_fix['elv_plan_id'].apply(lambda x: str(int(grace_period_map[x])))

C:\Users\james.richmond\AppData\Local\Temp\ipykernel_23952\3288914502.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grace_periods_to_fix['double_check'] = grace_periods_to_fix['elv_plan_id'].apply(lambda x: str(int(grace_period_map[x])))


In [102]:
# grace_periods_to_fix[['client_id', 'cu_plan_id', 'organization_id', 'elv_plan_id', 'grace_period', 'grace_period_fix', 'double_check']]

grace_periods_to_fix[grace_periods_to_fix['grace_period_fix'] != grace_periods_to_fix['grace_period']][['client_id', 'rmrcode', 'cu_plan_id', 'organization_id', 'elv_plan_id', 'grace_period', 'grace_period_fix', 'plan_coverage_config.grace_period_type.grace_period_type', 'double_check']]


,client_id,rmrcode,cu_plan_id,organization_id,elv_plan_id,grace_period,grace_period_fix,plan_coverage_config.grace_period_type.grace_period_type,double_check
3052,86877zuj5,RMRSAC,868c5gfc5,8908,44731,75,30,CUSTOM,30
3054,86877zuj5,RMRSAC,868c5gfe0,8908,44732,75,30,CUSTOM,30
3057,86877zuj5,RMRSAC,868g964wp,8908,96115,75,30,CUSTOM,30


In [103]:
grace_periods_to_fix[['client_id', 'cu_plan_id', 'organization_id', 'elv_plan_id', 'grace_period', 'grace_period_fix']]

,client_id,cu_plan_id,organization_id,elv_plan_id,grace_period,grace_period_fix
3052,86877zuj5,868c5gfc5,8908,44731,75,30
3054,86877zuj5,868c5gfe0,8908,44732,75,30
3057,86877zuj5,868g964wp,8908,96115,75,30


In [57]:
dc_df[[
    'id',
    'plan_coverage_config.grace_period_type.grace_period_days_amount',
    'plan_coverage_config.grace_period_type.grace_period_type'
]]


dc_df['double_check'] = dc_df.apply(
    lambda row:
        75 if row['plan_coverage_config.grace_period_type.grace_period_type'] == "TWO_AND_HALF_MONTH" else
        row['plan_coverage_config.grace_period_type.grace_period_days_amount'] if row['plan_coverage_config.grace_period_type.grace_period_type'] == "CUSTOM" else
        0 if row['plan_coverage_config.grace_period_type.grace_period_type'] == "DOES_NOT_APPLY"
        else pd.NA,
    axis=1
)


In [83]:
grace_periods_to_fix = pd.read_csv("grace_periods_to_fix.csv")

In [91]:
update_responses = []
for index, row in grace_periods_to_fix.iterrows():
    update_res = rcu.patch(
        task_id=row['plan_id'],
        field_id="2a105ee9-2dab-4e12-a6ed-bf5df55ee0c4",
        value=int(row['correct_grace_period'])
    )

    update_responses.append(update_res)

In [90]:
grace_periods_to_fix

,client_id,plan_id,name,current_grace_period,correct_grace_period
0,86877zquw,868g954ug,RMRNXT DCA 2026,0.0,75
1,86877zmcc,8687ac9ad,RMRECR DCA 2024,90.0,75
2,86877zmcc,868g9540w,RMRECR DCA 2026,0.0,75
3,86877ztdb,868g95450,RMRRCH FSA 2026,0.0,75
4,86877ztdb,868g9541r,RMRRCH DCA 2026,0.0,75
...,...,...,...,...,...
490,86877zrhp,868c5gbgt,RMRPGR FSA 2025,75.0,0
491,86877zrhp,868c5gbeb,RMRPGR DCA 2025,0.0,75
492,868dzycxh,868g9m72c,RMRFCP DCA 2026,0.0,75
493,868e27tne,868e27t9k,RMRPSD DCA 2025,0.0,75
